In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


# ----------------------------
# CONFIG
# ----------------------------

PARQUET_PATH = "your_file.parquet"

CFG = {
    "date_col": "date",
    "time_col": "timestamp",      # set to None if you want bar index on x-axis

    "open_col": "HA_Open",
    "high_col": "HA_High",
    "low_col": "HA_Low",
    "close_col": "HA_Close",

    "jma_col": "JMA",
    "colour_col": "colour",

    "bid_col": "bestBid",
    "ask_col": "bestAsk",

    "green_value": "green",
    "red_value": "red",

    "k": 0.0,                     # per execution cost
}


# ----------------------------
# LOAD DATA
# ----------------------------

df = pd.read_parquet(PARQUET_PATH)

sort_cols = [CFG["date_col"]]
if CFG["time_col"] is not None:
    sort_cols.append(CFG["time_col"])

df = df.sort_values(sort_cols).reset_index(drop=True)

In [ ]:
def add_entry_masks(day, cfg):
    high = day[cfg["high_col"]].to_numpy(float)
    low = day[cfg["low_col"]].to_numpy(float)
    jma = day[cfg["jma_col"]].to_numpy(float)
    colour = day[cfg["colour_col"]]

    mid = low + (high - low) / 2.0

    n = len(day)
    can_long = np.zeros(n, dtype=bool)
    can_short = np.zeros(n, dtype=bool)

    prev_green = colour.shift(1).eq(cfg["green_value"]).to_numpy()
    prev_red = colour.shift(1).eq(cfg["red_value"]).to_numpy()

    # At decision i, "previous bar crossed down" means:
    # mid[i-2] > jma[i-2] and mid[i-1] <= jma[i-1]
    cross_down_prev = np.zeros(n, dtype=bool)
    cross_up_prev = np.zeros(n, dtype=bool)

    cross_down_prev[2:] = (mid[:-2] > jma[:-2]) & (mid[1:-1] <= jma[1:-1])
    cross_up_prev[2:] = (mid[:-2] < jma[:-2]) & (mid[1:-1] >= jma[1:-1])

    can_long = prev_green & cross_down_prev
    can_short = prev_red & cross_up_prev

    out = day.copy()
    out["canEnterLong"] = can_long
    out["canEnterShort"] = can_short
    return out

In [ ]:
POS_SHORT = 0
POS_FLAT = 1
POS_LONG = 2

IDX_TO_POS = {
    POS_SHORT: -1,
    POS_FLAT: 0,
    POS_LONG: 1,
}

NEG_INF = -1e100


def optimize_day(day, cfg):
    day = add_entry_masks(day.reset_index(drop=True), cfg)

    bid = day[cfg["bid_col"]].to_numpy(float)
    ask = day[cfg["ask_col"]].to_numpy(float)
    can_long = day["canEnterLong"].to_numpy(bool)
    can_short = day["canEnterShort"].to_numpy(bool)
    k = float(cfg["k"])

    n = len(day)
    assert n >= 3
    assert np.isfinite(bid).all()
    assert np.isfinite(ask).all()
    assert (ask >= bid).all()

    v = np.full((n, 3), NEG_INF, dtype=float)
    prev_state = np.full((n, 3), -1, dtype=np.int8)
    prev_action = np.full((n, 3), "", dtype=object)
    prev_exec_price = np.full((n, 3), np.nan, dtype=float)

    v[0, POS_FLAT] = 0.0

    def relax(i, src, dst, cash_delta, action, exec_price=np.nan):
        val = v[i, src] + cash_delta
        j = i + 1
        if val > v[j, dst]:
            v[j, dst] = val
            prev_state[j, dst] = src
            prev_action[j, dst] = action
            prev_exec_price[j, dst] = exec_price

    # Decision at i, execution at i+1.
    # Last bar n-1 is preserved as final execution/close bar.
    for i in range(n - 1):
        j = i + 1

        # flat -> flat
        if v[i, POS_FLAT] > NEG_INF / 2:
            relax(i, POS_FLAT, POS_FLAT, 0.0, "stay_away")

            # Entries only if there will still be a later bar to close.
            if i <= n - 3:
                if can_long[i]:
                    relax(i, POS_FLAT, POS_LONG, -ask[j] - k, "enter_long", ask[j])

                if can_short[i]:
                    relax(i, POS_FLAT, POS_SHORT, bid[j] - k, "enter_short", bid[j])

        # long -> long / flat
        if v[i, POS_LONG] > NEG_INF / 2:
            relax(i, POS_LONG, POS_LONG, 0.0, "hold_long")
            relax(i, POS_LONG, POS_FLAT, bid[j] - k, "exit_long", bid[j])

        # short -> short / flat
        if v[i, POS_SHORT] > NEG_INF / 2:
            relax(i, POS_SHORT, POS_SHORT, 0.0, "hold_short")
            relax(i, POS_SHORT, POS_FLAT, -ask[j] - k, "exit_short", ask[j])

    # Forced end-of-day close if still in position.
    terminal_values = np.array([
        v[n - 1, POS_SHORT] - ask[n - 1] - k,
        v[n - 1, POS_FLAT],
        v[n - 1, POS_LONG] + bid[n - 1] - k,
    ])

    terminal_pre_state = int(np.argmax(terminal_values))
    terminal_cash = float(terminal_values[terminal_pre_state])

    forced_action = ""
    forced_price = np.nan

    if terminal_pre_state == POS_LONG:
        forced_action = "force_exit_long"
        forced_price = bid[n - 1]
    elif terminal_pre_state == POS_SHORT:
        forced_action = "force_exit_short"
        forced_price = ask[n - 1]

    # Backtrack DP path before possible forced close.
    state_after = np.empty(n, dtype=np.int8)
    action_at_decision = np.full(n, "", dtype=object)
    exec_price_at_decision = np.full(n, np.nan, dtype=float)

    state = terminal_pre_state
    state_after[n - 1] = state

    for j in range(n - 1, 0, -1):
        action_at_decision[j - 1] = prev_action[j, state]
        exec_price_at_decision[j - 1] = prev_exec_price[j, state]
        state = int(prev_state[j, state])
        state_after[j - 1] = state

    if forced_action:
        action_at_decision[n - 1] = forced_action
        exec_price_at_decision[n - 1] = forced_price

    position_after_bar = np.array([IDX_TO_POS[int(x)] for x in state_after], dtype=np.int8)

    out = day.copy()
    out["dp_cash_state_short"] = v[:, POS_SHORT]
    out["dp_cash_state_flat"] = v[:, POS_FLAT]
    out["dp_cash_state_long"] = v[:, POS_LONG]
    out["position_after_bar"] = position_after_bar
    out["action"] = action_at_decision
    out["execution_price"] = exec_price_at_decision
    out["terminal_cash"] = terminal_cash

    out, segments = build_position_labels_and_segments(out, cfg)

    return out, segments

In [ ]:
def build_position_labels_and_segments(day, cfg):
    n = len(day)
    k = float(cfg["k"])

    actions = day["action"].to_numpy()
    prices = day["execution_price"].to_numpy(float)

    position_label = np.zeros(n, dtype=np.int8)

    trade_segments = []

    open_side = 0
    entry_bar = None
    entry_price = np.nan
    entry_action_bar = None

    for action_bar, action in enumerate(actions):
        if not action:
            continue

        if action == "enter_long":
            open_side = 1
            entry_bar = action_bar + 1
            entry_action_bar = action_bar
            entry_price = prices[action_bar]

        elif action == "enter_short":
            open_side = -1
            entry_bar = action_bar + 1
            entry_action_bar = action_bar
            entry_price = prices[action_bar]

        elif action in ("exit_long", "exit_short"):
            exit_bar = action_bar + 1
            exit_price = prices[action_bar]

            position_label[entry_bar:exit_bar + 1] = open_side

            if open_side == 1:
                net_pnl = exit_price - entry_price - 2.0 * k
            else:
                net_pnl = entry_price - exit_price - 2.0 * k

            trade_segments.append({
                "side": open_side,
                "startBar": entry_bar,
                "endBar": exit_bar,
                "entryDecisionBar": entry_action_bar,
                "exitDecisionBar": action_bar,
                "entryPrice": entry_price,
                "exitPrice": exit_price,
                "netPnl": net_pnl,
            })

            open_side = 0
            entry_bar = None
            entry_price = np.nan
            entry_action_bar = None

        elif action in ("force_exit_long", "force_exit_short"):
            exit_bar = action_bar
            exit_price = prices[action_bar]

            position_label[entry_bar:exit_bar + 1] = open_side

            if open_side == 1:
                net_pnl = exit_price - entry_price - 2.0 * k
            else:
                net_pnl = entry_price - exit_price - 2.0 * k

            trade_segments.append({
                "side": open_side,
                "startBar": entry_bar,
                "endBar": exit_bar,
                "entryDecisionBar": entry_action_bar,
                "exitDecisionBar": action_bar,
                "entryPrice": entry_price,
                "exitPrice": exit_price,
                "netPnl": net_pnl,
            })

            open_side = 0
            entry_bar = None
            entry_price = np.nan
            entry_action_bar = None

    day = day.copy()
    day["position_label"] = position_label

    segments = extract_all_segments(day, trade_segments)
    return day, segments


def extract_all_segments(day, trade_segments):
    pos = day["position_label"].to_numpy()
    n = len(pos)

    trade_by_range = {
        (seg["startBar"], seg["endBar"], seg["side"]): seg
        for seg in trade_segments
    }

    rows = []
    seg_id = 0
    start = 0
    current = int(pos[0])

    for i in range(1, n + 1):
        if i == n or int(pos[i]) != current:
            end = i - 1

            row = {
                "segmentId": seg_id,
                "side": current,
                "kind": "long" if current == 1 else "short" if current == -1 else "bad",
                "startBar": start,
                "endBar": end,
                "lengthBars": end - start + 1,
                "entryDecisionBar": np.nan,
                "exitDecisionBar": np.nan,
                "entryPrice": np.nan,
                "exitPrice": np.nan,
                "netPnl": np.nan,
            }

            key = (start, end, current)
            if key in trade_by_range:
                trade = trade_by_range[key]
                row.update({
                    "entryDecisionBar": trade["entryDecisionBar"],
                    "exitDecisionBar": trade["exitDecisionBar"],
                    "entryPrice": trade["entryPrice"],
                    "exitPrice": trade["exitPrice"],
                    "netPnl": trade["netPnl"],
                })

            rows.append(row)

            seg_id += 1
            if i < n:
                start = i
                current = int(pos[i])

    return pd.DataFrame(rows)

In [ ]:
def run_dp_all_days(df, cfg):
    all_days = []
    all_segments = []

    for date_value, day in df.groupby(cfg["date_col"], sort=False):
        day_out, seg = optimize_day(day, cfg)

        day_out[cfg["date_col"]] = date_value
        seg[cfg["date_col"]] = date_value

        all_days.append(day_out)
        all_segments.append(seg)

    out = pd.concat(all_days, ignore_index=True)
    segments = pd.concat(all_segments, ignore_index=True)

    cols = [cfg["date_col"], "segmentId", "kind", "side", "startBar", "endBar", "lengthBars",
            "entryDecisionBar", "exitDecisionBar", "entryPrice", "exitPrice", "netPnl"]

    return out, segments[cols]


dp_df, segments_df = run_dp_all_days(df, CFG)

print(dp_df[[CFG["date_col"], CFG["time_col"], "position_label", "action", "execution_price"]].head(20))
print(segments_df.head(20))

In [ ]:
def plot_dp_day(dp_df, segments_df, cfg, date_value, show_bad=True, height=750):
    day = dp_df[dp_df[cfg["date_col"]] == date_value].reset_index(drop=True)
    segs = segments_df[segments_df[cfg["date_col"]] == date_value].reset_index(drop=True)

    if cfg["time_col"] is None:
        x = np.arange(len(day))
        bar_delta = 1.0
    else:
        x = pd.to_datetime(day[cfg["time_col"]])
        diffs = x.diff().dropna()
        bar_delta = diffs.median() if len(diffs) else pd.Timedelta(seconds=1)

    fig = go.Figure()

    fig.add_trace(go.Candlestick(
        x=x,
        open=day[cfg["open_col"]],
        high=day[cfg["high_col"]],
        low=day[cfg["low_col"]],
        close=day[cfg["close_col"]],
        name="HA OHLC",
        increasing_line_color="green",
        decreasing_line_color="red",
    ))

    fig.add_trace(go.Scatter(
        x=x,
        y=day[cfg["jma_col"]],
        mode="lines",
        name="JMA",
        line=dict(width=1.5, color="blue"),
    ))

    for _, seg in segs.iterrows():
        side = int(seg["side"])

        if side == 0 and not show_bad:
            continue

        s = int(seg["startBar"])
        e = int(seg["endBar"])

        if cfg["time_col"] is None:
            x0 = s - 0.5
            x1 = e + 0.5
        else:
            x0 = x.iloc[s] - bar_delta / 2
            x1 = x.iloc[e] + bar_delta / 2

        if side == 1:
            color = "rgba(0, 180, 0, 0.14)"
        elif side == -1:
            color = "rgba(220, 0, 0, 0.14)"
        else:
            color = "rgba(120, 120, 120, 0.06)"

        fig.add_vrect(
            x0=x0,
            x1=x1,
            fillcolor=color,
            line_width=0,
            layer="below",
        )

    entries_x = []
    entries_y = []
    entries_text = []

    exits_x = []
    exits_y = []
    exits_text = []

    for _, seg in segs[segs["side"] != 0].iterrows():
        entry_bar = int(seg["startBar"])
        exit_bar = int(seg["endBar"])

        entries_x.append(x.iloc[entry_bar] if cfg["time_col"] is not None else entry_bar)
        entries_y.append(float(seg["entryPrice"]))
        entries_text.append(f"entry {seg['kind']} pnl={seg['netPnl']:.2f}")

        exits_x.append(x.iloc[exit_bar] if cfg["time_col"] is not None else exit_bar)
        exits_y.append(float(seg["exitPrice"]))
        exits_text.append(f"exit {seg['kind']} pnl={seg['netPnl']:.2f}")

    fig.add_trace(go.Scatter(
        x=entries_x,
        y=entries_y,
        mode="markers",
        name="entries",
        text=entries_text,
        hovertemplate="%{text}<br>x=%{x}<br>price=%{y}<extra></extra>",
        marker=dict(symbol="triangle-up", size=10, color="black"),
    ))

    fig.add_trace(go.Scatter(
        x=exits_x,
        y=exits_y,
        mode="markers",
        name="exits",
        text=exits_text,
        hovertemplate="%{text}<br>x=%{x}<br>price=%{y}<extra></extra>",
        marker=dict(symbol="x", size=9, color="black"),
    ))

    fig.update_layout(
        title=f"DP audit — {date_value}",
        height=height,
        xaxis_rangeslider_visible=False,
        hovermode="x unified",
        legend=dict(orientation="h"),
        margin=dict(l=40, r=20, t=60, b=40),
    )

    fig.show()

In [ ]:
plot_dp_day(dp_df, segments_df, CFG, "2025-10-10")